In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 19.1 - Overview, paths, and fixed ablation design
# Purpose:
# Begin coarse-to-fine ablation of the significant collective whole-chromosome
# association reconstructed in Notebook 18.
#
# Notebook 18 result:
# - all 1,287,844 variable chromosomal unitigs together;
# - whole-sequence variance fraction = 0.538815;
# - empirical p = 0.000999 from 1,000 permutations.
#
# This notebook asks which BROAD groups of unitigs can be removed while
# retaining that association, and which removals weaken it.
#
# To keep the first ablation unbiased by MIC:
# - unitigs are mapped to the fixed E. coli MG1655 reference chromosome;
# - uniquely mapped unitigs are divided into 10 equal-width reference windows;
# - multi-mapped unitigs form one additional group;
# - unmapped unitigs form one additional group.
#
# Thus all variable unitigs are assigned to exactly one of 12 broad groups.
#
# For each group this notebook calculates:
# A. leave-one-group-out: all unitigs except that group;
# B. group-only: that group by itself.
#
# Each kernel uses the same frequency-centred construction as Notebook 18.
# Each observed variance fraction is tested by 1,000 MIC permutations.
#
# This is the first coarse ablation only.
# It does not yet remove individual unitigs or claim a causal combination.

from pathlib import Path
import gzip
import json
import os
import re
import shutil
import subprocess
import time

import numpy as np
import pandas as pd
from scipy import sparse, optimize, stats
from IPython.display import display
PROJECT_ROOT = _repo_root()
NOTEBOOK_DIR = PROJECT_ROOT / "03_Notebooks" / "04_Genome_Comparison"
RESULTS_TABLE_DIR = PROJECT_ROOT / "05_Results" / "Tables"

UNITIG_DIR = PROJECT_ROOT / "04_Intermediate" / "10_Whole_Chromosome_Unitigs"
UNITIG_FASTA = UNITIG_DIR / "10_variable_unitigs.fasta.gz"
UNITIG_MATRIX = UNITIG_DIR / "10_variable_unitig_matrix_176xM.npz"
UNITIG_SAMPLES = UNITIG_DIR / "10_unitig_sample_order.csv"
NB10_QC = RESULTS_TABLE_DIR / "10_unitig_representation_final_QC.csv"

NB18_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "18_Collective_Whole_Chromosome_Association"
)

NB18_K = NB18_DIR / "18_whole_sequence_unitig_similarity_matrix.npz"
NB18_SUMMARY = RESULTS_TABLE_DIR / "18_collective_unitig_association_summary.csv"
NB18_QC = RESULTS_TABLE_DIR / "18_collective_unitig_association_final_QC.csv"

# Fixed reference already used in the previous ceftazidime project.
REFERENCE_FASTA = (
    MYDRIVE
    / "Genome_MIC_AMR_Emergence"
    / "04_Population_Structure"
    / "Notebook04"
    / "checkpoints"
    / "reference"
    / "04_MG1655_reference.fna"
)

NB19_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "19_Broad_Unitig_Ablation"
)

NB19_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_ASSIGNMENT = NB19_DIR / "19_unitig_reference_assignment.npz"
GROUP_KERNEL_COMPONENTS = NB19_DIR / "19_group_kernel_components.npz"
GROUP_MANIFEST = RESULTS_TABLE_DIR / "19_broad_ablation_group_manifest.csv"
MAPPING_SUMMARY = RESULTS_TABLE_DIR / "19_unitig_reference_mapping_summary.csv"
OBSERVED_ABLATION = RESULTS_TABLE_DIR / "19_broad_ablation_observed_results.csv"
PERMUTATION_RESULTS = NB19_DIR / "19_broad_ablation_permutation_results.csv.gz"
FINAL_RESULTS = RESULTS_TABLE_DIR / "19_broad_ablation_final_results.csv"
FINAL_QC = RESULTS_TABLE_DIR / "19_broad_ablation_final_QC.csv"
COMPLETION_FILE = NB19_DIR / "19_BROAD_UNITIG_ABLATION_COMPLETE.json"

EXPECTED_PATHOGENS = 176
EXPECTED_UNITIGS = 1_287_844
N_REFERENCE_WINDOWS = 10
N_PERMUTATIONS = 1000
PERMUTATION_SEED = 20260914

for path in [
    PROJECT_ROOT,
    NOTEBOOK_DIR,
    RESULTS_TABLE_DIR,
    UNITIG_FASTA,
    UNITIG_MATRIX,
    UNITIG_SAMPLES,
    NB10_QC,
    NB18_K,
    NB18_SUMMARY,
    NB18_QC,
    REFERENCE_FASTA,
]:
    assert path.exists(), f"Required input not found: {path}"

print("Notebook 19 - Broad Unitig Ablation of the Collective Whole-Chromosome Association")
print("Pathogens:", EXPECTED_PATHOGENS)
print("Variable unitigs:", f"{EXPECTED_UNITIGS:,}")
print("Reference windows:", N_REFERENCE_WINDOWS)
print("Additional groups: multi-mapped and unmapped")
print("Permutations per tested kernel:", N_PERMUTATIONS)
print("No individual-unitig ablation is performed here.")
print("\nTransition: Cell 19.2 will verify Notebook 18, the unitig representation, and the fixed MG1655 reference.")


In [ ]:
#@title Cell 19.2 - Verify Notebook 18, unitigs, phenotype, and MG1655 reference
# Purpose:
# Confirm that Notebook 18 passed, recover the exact 176-pathogen phenotype
# order, and verify the reference chromosome used for broad spatial grouping.

nb10_qc = pd.read_csv(NB10_QC)
nb18_qc = pd.read_csv(NB18_QC)
nb18_summary = pd.read_csv(NB18_SUMMARY)

assert len(nb10_qc) == 1
assert len(nb18_qc) == 1
assert len(nb18_summary) == 1

assert bool(nb10_qc.loc[0, "final_QC_pass"])
assert bool(nb18_qc.loc[0, "final_QC_pass"])

assert int(nb10_qc.loc[0, "variable_unitigs_M"]) == EXPECTED_UNITIGS
assert int(nb18_summary.loc[0, "variable_unitigs_used_together"]) == EXPECTED_UNITIGS

baseline_variance_fraction = float(
    nb18_summary.loc[
        0,
        "whole_sequence_unitig_variance_fraction",
    ]
)

baseline_empirical_p = float(
    nb18_summary.loc[
        0,
        "empirical_p_value",
    ]
)

assert abs(
    baseline_variance_fraction
    - 0.538815
) < 0.001

samples = pd.read_csv(
    UNITIG_SAMPLES
)

required_sample_columns = {
    "sample_index",
    "biosample",
    "assembly_accession",
    "log2_mic",
}

missing = (
    required_sample_columns
    - set(samples.columns)
)

assert not missing, (
    "Sample-order file missing columns: "
    + ", ".join(sorted(missing))
)

samples = (
    samples
    .sort_values("sample_index")
    .reset_index(drop=True)
)

assert len(samples) == EXPECTED_PATHOGENS
assert samples["biosample"].nunique() == EXPECTED_PATHOGENS
assert np.array_equal(
    samples["sample_index"].to_numpy(dtype=int),
    np.arange(EXPECTED_PATHOGENS),
)

y = samples["log2_mic"].to_numpy(dtype=float)

assert y.shape == (EXPECTED_PATHOGENS,)
assert np.isfinite(y).all()

matrix_shape = sparse.load_npz(
    UNITIG_MATRIX
).shape

assert matrix_shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_UNITIGS,
)

# Load the Notebook 18 full unitig kernel.
with np.load(NB18_K) as archive:
    assert "K_unitig" in archive.files
    K_full = np.asarray(
        archive["K_unitig"],
        dtype=float,
    )

assert K_full.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_PATHOGENS,
)

K_full = (
    K_full
    + K_full.T
) / 2.0

# Read the fixed MG1655 reference FASTA without requiring Biopython.
reference_records = []
current_name = None
current_length = 0

with open(
    REFERENCE_FASTA,
    "r",
    encoding="utf-8",
) as handle:
    for raw_line in handle:
        line = raw_line.strip()

        if not line:
            continue

        if line.startswith(">"):
            if current_name is not None:
                reference_records.append(
                    (
                        current_name,
                        current_length,
                    )
                )

            current_name = line[1:].split()[0]
            current_length = 0

        else:
            current_length += len(line)

if current_name is not None:
    reference_records.append(
        (
            current_name,
            current_length,
        )
    )

assert len(reference_records) == 1, (
    "Expected exactly one chromosome in the fixed MG1655 reference FASTA; "
    f"found {len(reference_records)}."
)

reference_name, reference_length = reference_records[0]

assert reference_length > 4_000_000
assert reference_length < 6_000_000

print("Notebook 10 QC: PASS")
print("Notebook 18 QC: PASS")
print("Baseline whole-sequence variance fraction:", baseline_variance_fraction)
print("Baseline empirical p-value:", baseline_empirical_p)
print("Unitig matrix shape:", matrix_shape)
print("MG1655 reference record:", reference_name)
print("MG1655 reference length:", f"{reference_length:,}", "bp")
print("Continuous log2 MIC range:", float(y.min()), "to", float(y.max()))

print("\nCell 19.2 complete.")
print("Transition: Cell 19.3 will map the 1,287,844 unitigs to MG1655 for broad, MIC-independent grouping.")


In [ ]:
#@title Cell 19.3 - Map all variable unitigs to the MG1655 chromosome
# Purpose:
# Map each variable unitig to the fixed MG1655 chromosome only to assign
# broad reference position.
#
# Bowtie2 is run end-to-end with up to two reported alignments:
# - exactly one reported alignment -> uniquely mapped;
# - two reported alignments -> multi-mapped;
# - no reported alignment -> unmapped.
#
# Mapping is used only for coarse grouping. It is not variant calling.
#
# Large temporary files remain in /content and are removed after assignment.

LOCAL_WORK = Path("/content/nb19_unitig_mapping")
LOCAL_WORK.mkdir(parents=True, exist_ok=True)

LOCAL_REFERENCE = LOCAL_WORK / "MG1655_reference.fna"
LOCAL_UNITIG_FASTA = LOCAL_WORK / "variable_unitigs.fasta.gz"
BOWTIE2_INDEX_PREFIX = LOCAL_WORK / "mg1655"
SAM_FILE = LOCAL_WORK / "unitigs_to_mg1655.sam"

# Install Bowtie2 only in the temporary Colab runtime.
if shutil.which("bowtie2") is None:
    print("Installing Bowtie2 in the temporary Colab runtime...")

    subprocess.run(
        [
            "apt-get",
            "-qq",
            "update",
        ],
        check=True,
    )

    subprocess.run(
        [
            "apt-get",
            "-qq",
            "install",
            "-y",
            "bowtie2",
        ],
        check=True,
    )

version_result = subprocess.run(
    [
        "bowtie2",
        "--version",
    ],
    check=True,
    capture_output=True,
    text=True,
)

bowtie2_version = version_result.stdout.splitlines()[0].strip()

print("Bowtie2:", bowtie2_version)

# Copy the small reference and the compressed unitig FASTA to local storage.
if (
    not LOCAL_REFERENCE.exists()
    or LOCAL_REFERENCE.stat().st_size
    != REFERENCE_FASTA.stat().st_size
):
    shutil.copy2(
        REFERENCE_FASTA,
        LOCAL_REFERENCE,
    )

if (
    not LOCAL_UNITIG_FASTA.exists()
    or LOCAL_UNITIG_FASTA.stat().st_size
    != UNITIG_FASTA.stat().st_size
):
    print("Copying unitig FASTA to temporary local storage...")

    shutil.copy2(
        UNITIG_FASTA,
        LOCAL_UNITIG_FASTA,
    )

# Build the Bowtie2 index once.
index_marker = Path(
    str(BOWTIE2_INDEX_PREFIX)
    + ".1.bt2"
)

large_index_marker = Path(
    str(BOWTIE2_INDEX_PREFIX)
    + ".1.bt2l"
)

if not (
    index_marker.exists()
    or large_index_marker.exists()
):
    print("Building MG1655 Bowtie2 index...")

    subprocess.run(
        [
            "bowtie2-build",
            str(LOCAL_REFERENCE),
            str(BOWTIE2_INDEX_PREFIX),
        ],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )

if not SAM_FILE.exists():
    print("Mapping all variable unitigs...")

    mapping_start = time.time()

    command = [
        "bowtie2",
        "-f",
        "-x",
        str(BOWTIE2_INDEX_PREFIX),
        "-U",
        str(LOCAL_UNITIG_FASTA),
        "--very-sensitive",
        "-k",
        "2",
        "--no-unal",
        "--threads",
        str(
            max(
                1,
                min(
                    2,
                    os.cpu_count() or 1,
                ),
            )
        ),
        "-S",
        str(SAM_FILE),
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Bowtie2 mapping failed.\n"
            + result.stderr[-5000:]
        )

    print(
        result.stderr.strip().splitlines()[-5:]
    )

    print(
        "Mapping elapsed:",
        f"{(time.time() - mapping_start) / 60.0:.1f} minutes",
    )

else:
    print("Existing temporary SAM file found; mapping step skipped.")

assert SAM_FILE.exists()
assert SAM_FILE.stat().st_size > 0

print("Temporary SAM size:", f"{SAM_FILE.stat().st_size / 1024**2:.1f} MiB")
print("\nCell 19.3 complete.")
print("Transition: Cell 19.4 will classify every unitig into 10 reference windows, multi-mapped, or unmapped.")


In [ ]:
#@title Cell 19.4 - Assign every unitig to one of 12 broad ablation groups
# Purpose:
# Convert the mapping output into a complete mutually exclusive partition:
# - W01-W10: one unique MG1655 reference window;
# - MULTIMAPPED;
# - UNMAPPED.
#
# All 1,287,844 variable unitigs must be assigned exactly once.

WINDOW_EDGES = np.linspace(
    0,
    reference_length,
    N_REFERENCE_WINDOWS + 1,
    dtype=np.int64,
)

MULTIMAPPED_CODE = N_REFERENCE_WINDOWS
UNMAPPED_CODE = N_REFERENCE_WINDOWS + 1
N_GROUPS = N_REFERENCE_WINDOWS + 2

alignment_count = np.zeros(
    EXPECTED_UNITIGS,
    dtype=np.uint8,
)

reference_midpoint = np.full(
    EXPECTED_UNITIGS,
    -1,
    dtype=np.int32,
)

def unitig_id_to_index(qname):
    match = re.fullmatch(
        r"U(\d+)",
        qname,
    )

    if match is None:
        raise ValueError(
            f"Unexpected unitig identifier in SAM: {qname}"
        )

    index = int(
        match.group(1)
    ) - 1

    if not (
        0
        <= index
        < EXPECTED_UNITIGS
    ):
        raise ValueError(
            f"Unitig index out of range: {qname}"
        )

    return index

def cigar_reference_span(cigar):
    if cigar == "*":
        return 0

    span = 0

    for length_text, operation in re.findall(
        r"(\d+)([MIDNSHP=X])",
        cigar,
    ):
        if operation in {
            "M",
            "D",
            "N",
            "=",
            "X",
        }:
            span += int(
                length_text
            )

    return span

print("Parsing temporary SAM...")

parse_start = time.time()

with open(
    SAM_FILE,
    "r",
    encoding="utf-8",
    errors="replace",
) as handle:
    for raw_line in handle:
        if raw_line.startswith("@"):
            continue

        fields = raw_line.rstrip(
            "\n"
        ).split(
            "\t"
        )

        if len(fields) < 6:
            raise ValueError(
                "Unexpected SAM record with fewer than 6 fields."
            )

        qname = fields[0]
        flag = int(fields[1])

        if flag & 0x4:
            continue

        index = unitig_id_to_index(
            qname
        )

        if alignment_count[index] < 2:
            alignment_count[index] += 1

        if alignment_count[index] == 1:
            position_1_based = int(
                fields[3]
            )

            cigar = fields[5]

            reference_span = cigar_reference_span(
                cigar
            )

            assert reference_span > 0

            midpoint_0_based = int(
                round(
                    (
                        position_1_based
                        - 1
                    )
                    + (
                        reference_span
                        - 1
                    )
                    / 2.0
                )
            )

            midpoint_0_based = min(
                max(
                    midpoint_0_based,
                    0,
                ),
                reference_length - 1,
            )

            reference_midpoint[
                index
            ] = midpoint_0_based

group_code = np.full(
    EXPECTED_UNITIGS,
    UNMAPPED_CODE,
    dtype=np.uint8,
)

unique_mask = (
    alignment_count == 1
)

multi_mask = (
    alignment_count >= 2
)

unique_window_codes = np.searchsorted(
    WINDOW_EDGES[1:],
    reference_midpoint[
        unique_mask
    ],
    side="right",
).astype(
    np.uint8
)

assert unique_window_codes.min() >= 0
assert unique_window_codes.max() < N_REFERENCE_WINDOWS

group_code[
    unique_mask
] = unique_window_codes

group_code[
    multi_mask
] = MULTIMAPPED_CODE

assert np.all(
    (
        group_code >= 0
    )
    & (
        group_code < N_GROUPS
    )
)

group_names = [
    f"W{window_index + 1:02d}"
    for window_index in range(
        N_REFERENCE_WINDOWS
    )
] + [
    "MULTIMAPPED",
    "UNMAPPED",
]

group_rows = []

for group_index, group_name in enumerate(
    group_names
):
    mask = (
        group_code
        == group_index
    )

    n_unitigs = int(
        mask.sum()
    )

    if group_index < N_REFERENCE_WINDOWS:
        start_0 = int(
            WINDOW_EDGES[
                group_index
            ]
        )

        end_0_exclusive = int(
            WINDOW_EDGES[
                group_index + 1
            ]
        )

        group_type = "unique_reference_window"

    elif group_index == MULTIMAPPED_CODE:
        start_0 = np.nan
        end_0_exclusive = np.nan
        group_type = "multi_mapped"

    else:
        start_0 = np.nan
        end_0_exclusive = np.nan
        group_type = "unmapped"

    group_rows.append(
        {
            "group_code": group_index,
            "group_name": group_name,
            "group_type": group_type,
            "reference_start_0_based": start_0,
            "reference_end_0_based_exclusive": end_0_exclusive,
            "n_unitigs": n_unitigs,
            "fraction_of_all_unitigs": (
                n_unitigs
                / EXPECTED_UNITIGS
            ),
        }
    )

group_manifest = pd.DataFrame(
    group_rows
)

assert int(
    group_manifest[
        "n_unitigs"
    ].sum()
) == EXPECTED_UNITIGS

np.savez_compressed(
    REFERENCE_ASSIGNMENT,
    group_code=group_code,
    alignment_count=alignment_count,
    reference_midpoint=reference_midpoint,
    window_edges=WINDOW_EDGES,
)

group_manifest.to_csv(
    GROUP_MANIFEST,
    index=False,
)

mapping_summary = pd.DataFrame(
    [
        {
            "total_unitigs": EXPECTED_UNITIGS,
            "uniquely_mapped_unitigs": int(
                unique_mask.sum()
            ),
            "multi_mapped_unitigs": int(
                multi_mask.sum()
            ),
            "unmapped_unitigs": int(
                (
                    alignment_count == 0
                ).sum()
            ),
            "uniquely_mapped_fraction": float(
                unique_mask.mean()
            ),
            "multi_mapped_fraction": float(
                multi_mask.mean()
            ),
            "unmapped_fraction": float(
                (
                    alignment_count == 0
                ).mean()
            ),
            "reference_name": reference_name,
            "reference_length_bp": reference_length,
        }
    ]
)

mapping_summary.to_csv(
    MAPPING_SUMMARY,
    index=False,
)

print("Unitig assignment: PASS")
display(mapping_summary)

print("\nBroad groups:")
display(group_manifest)

print(
    "\nAssignment parsing elapsed:",
    f"{(time.time() - parse_start) / 60.0:.1f} minutes",
)

# Remove the large temporary SAM and local unitig copy after assignments
# have been saved safely to project storage.
if SAM_FILE.exists():
    SAM_FILE.unlink()

if LOCAL_UNITIG_FASTA.exists():
    LOCAL_UNITIG_FASTA.unlink()

print("Large temporary mapping files deleted.")
print("\nCell 19.4 complete.")
print("Transition: Cell 19.5 will decompose the full unitig kernel into the 12 broad group contributions and verify exact reconstruction of Notebook 18 K.")


In [ ]:
#@title Cell 19.5 - Decompose the full unitig kernel into 12 group contributions
# Purpose:
# For each broad group, calculate its centred cross-product numerator and
# frequency denominator. Their sum must reconstruct the exact Notebook 18
# whole-unitig K.
#
# This makes leave-one-group-out kernels exact and efficient.

X = sparse.load_npz(
    UNITIG_MATRIX
).tocsc()

assert X.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_UNITIGS,
)

group_numerators = np.zeros(
    (
        N_GROUPS,
        EXPECTED_PATHOGENS,
        EXPECTED_PATHOGENS,
    ),
    dtype=np.float64,
)

group_denominators = np.zeros(
    N_GROUPS,
    dtype=np.float64,
)

group_unitig_counts = np.zeros(
    N_GROUPS,
    dtype=np.int64,
)

decomposition_start = time.time()

for group_index, group_name in enumerate(
    group_names
):
    columns = np.flatnonzero(
        group_code
        == group_index
    )

    group_unitig_counts[
        group_index
    ] = len(
        columns
    )

    assert len(columns) > 0, (
        f"Broad group {group_name} contains no unitigs."
    )

    X_group = X[
        :,
        columns,
    ]

    presence_counts = np.asarray(
        X_group.sum(
            axis=0
        )
    ).ravel().astype(
        np.float64
    )

    p_group = (
        presence_counts
        / EXPECTED_PATHOGENS
    )

    denominator_group = float(
        np.sum(
            p_group
            * (
                1.0
                - p_group
            )
        )
    )

    assert denominator_group > 0

    XX_group = (
        X_group.astype(
            np.int32
        )
        @ X_group.astype(
            np.int32
        ).T
    ).toarray().astype(
        np.float64
    )

    Xp_group = np.asarray(
        X_group
        @ p_group
    ).reshape(-1).astype(
        np.float64
    )

    p2_group = float(
        p_group
        @ p_group
    )

    numerator_group = (
        XX_group
        - Xp_group[:, None]
        - Xp_group[None, :]
        + p2_group
    )

    numerator_group = (
        numerator_group
        + numerator_group.T
    ) / 2.0

    group_numerators[
        group_index
    ] = numerator_group

    group_denominators[
        group_index
    ] = denominator_group

    print(
        group_name,
        "- unitigs:",
        f"{len(columns):,}",
        "- denominator:",
        f"{denominator_group:.3f}",
    )

total_numerator = np.sum(
    group_numerators,
    axis=0,
)

total_denominator = float(
    np.sum(
        group_denominators
    )
)

K_reconstructed = (
    total_numerator
    / total_denominator
)

K_reconstructed = (
    K_reconstructed
    + K_reconstructed.T
) / 2.0

max_reconstruction_error = float(
    np.max(
        np.abs(
            K_reconstructed
            - K_full
        )
    )
)

assert max_reconstruction_error < 1e-10, (
    "Broad-group decomposition did not reconstruct Notebook 18 K exactly enough."
)

np.savez_compressed(
    GROUP_KERNEL_COMPONENTS,
    group_numerators=group_numerators,
    group_denominators=group_denominators,
    group_unitig_counts=group_unitig_counts,
)

group_manifest = group_manifest.copy()

group_manifest[
    "kernel_denominator_contribution"
] = group_denominators

group_manifest[
    "kernel_denominator_fraction"
] = (
    group_denominators
    / total_denominator
)

group_manifest.to_csv(
    GROUP_MANIFEST,
    index=False,
)

print("\nExact K reconstruction: PASS")
print("Maximum absolute reconstruction error:", max_reconstruction_error)
print("Decomposition elapsed:", f"{(time.time() - decomposition_start) / 60.0:.1f} minutes")

display(
    group_manifest[
        [
            "group_name",
            "group_type",
            "n_unitigs",
            "fraction_of_all_unitigs",
            "kernel_denominator_fraction",
        ]
    ]
)

print("\nCell 19.5 complete.")
print("Transition: Cell 19.6 will fit the observed leave-one-group-out and group-only kernels.")


In [ ]:
#@title Cell 19.6 - Fit observed leave-one-group-out and group-only kernels
# Purpose:
# Measure how the collective variance fraction changes when each broad group
# is removed, and how much signal each broad group carries by itself.
#
# A large fall after removal means that broad group is a candidate contributor.
# A strong group-only association means that group independently carries
# sequence structure related to MIC.
#
# These are screening results for later refinement, not causal claims.

def prepare_kernel(K):
    K = np.asarray(
        K,
        dtype=float,
    )

    K = (
        K
        + K.T
    ) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(
        K
    )

    minimum_eigenvalue = float(
        eigenvalues.min()
    )

    if minimum_eigenvalue < -1e-6:
        raise ValueError(
            f"Kernel is not positive semidefinite: minimum eigenvalue = {minimum_eigenvalue}"
        )

    eigenvalues = np.maximum(
        eigenvalues,
        0.0,
    )

    transformed_intercept = (
        eigenvectors.T
        @ np.ones(
            K.shape[0],
            dtype=float,
        )
    )

    return {
        "K": K,
        "eigenvalues": eigenvalues,
        "eigenvectors": eigenvectors,
        "transformed_intercept": transformed_intercept,
        "minimum_eigenvalue": minimum_eigenvalue,
    }

def fit_null_reml_prepared(y_input, prepared):
    y_input = np.asarray(
        y_input,
        dtype=float,
    ).reshape(-1)

    eigenvalues = prepared[
        "eigenvalues"
    ]

    eigenvectors = prepared[
        "eigenvectors"
    ]

    transformed_intercept = prepared[
        "transformed_intercept"
    ]

    transformed_y = (
        eigenvectors.T
        @ y_input
    )

    n = len(
        y_input
    )

    degrees_of_freedom = (
        n - 1
    )

    def evaluate_ratio(ratio):
        if ratio < 0:
            return None

        covariance_eigenvalues = (
            1.0
            + ratio
            * eigenvalues
        )

        if np.any(
            covariance_eigenvalues <= 0
        ):
            return None

        inverse_weights = (
            1.0
            / covariance_eigenvalues
        )

        information = float(
            np.sum(
                transformed_intercept
                * transformed_intercept
                * inverse_weights
            )
        )

        if information <= 0:
            return None

        beta_0 = float(
            np.sum(
                transformed_intercept
                * transformed_y
                * inverse_weights
            )
            / information
        )

        transformed_residual = (
            transformed_y
            - beta_0
            * transformed_intercept
        )

        residual_quadratic = float(
            np.sum(
                transformed_residual
                * transformed_residual
                * inverse_weights
            )
        )

        if residual_quadratic <= 0:
            return None

        sigma_e2 = (
            residual_quadratic
            / degrees_of_freedom
        )

        sigma_g2 = (
            ratio
            * sigma_e2
        )

        objective = 0.5 * (
            degrees_of_freedom
            * np.log(
                sigma_e2
            )
            + np.log(
                covariance_eigenvalues
            ).sum()
            + np.log(
                information
            )
        )

        return {
            "objective": float(
                objective
            ),
            "ratio": float(
                ratio
            ),
            "sigma_g2": float(
                sigma_g2
            ),
            "sigma_e2": float(
                sigma_e2
            ),
            "variance_fraction": float(
                sigma_g2
                / (
                    sigma_g2
                    + sigma_e2
                )
            ),
        }

    def objective_on_log_ratio(log_ratio):
        result = evaluate_ratio(
            np.exp(
                log_ratio
            )
        )

        if result is None:
            return np.inf

        return result[
            "objective"
        ]

    optimized = optimize.minimize_scalar(
        objective_on_log_ratio,
        bounds=(
            -12.0,
            12.0,
        ),
        method="bounded",
        options={
            "xatol": 1e-8,
            "maxiter": 500,
        },
    )

    candidates = []

    zero_result = evaluate_ratio(
        0.0
    )

    if zero_result is not None:
        candidates.append(
            zero_result
        )

    if optimized.success:
        optimized_result = evaluate_ratio(
            float(
                np.exp(
                    optimized.x
                )
            )
        )

        if optimized_result is not None:
            candidates.append(
                optimized_result
            )

    high_result = evaluate_ratio(
        float(
            np.exp(
                12.0
            )
        )
    )

    if high_result is not None:
        candidates.append(
            high_result
        )

    assert candidates

    return min(
        candidates,
        key=lambda item: item[
            "objective"
        ],
    )

kernel_records = []
prepared_kernels = {}

for group_index, group_name in enumerate(
    group_names
):
    numerator_group = group_numerators[
        group_index
    ]

    denominator_group = float(
        group_denominators[
            group_index
        ]
    )

    # Group-only kernel.
    K_only = (
        numerator_group
        / denominator_group
    )

    K_only = (
        K_only
        + K_only.T
    ) / 2.0

    # Leave-one-group-out kernel.
    denominator_minus = (
        total_denominator
        - denominator_group
    )

    assert denominator_minus > 0

    K_minus = (
        total_numerator
        - numerator_group
    ) / denominator_minus

    K_minus = (
        K_minus
        + K_minus.T
    ) / 2.0

    prepared_only = prepare_kernel(
        K_only
    )

    prepared_minus = prepare_kernel(
        K_minus
    )

    fit_only = fit_null_reml_prepared(
        y,
        prepared_only,
    )

    fit_minus = fit_null_reml_prepared(
        y,
        prepared_minus,
    )

    upper = np.triu_indices(
        EXPECTED_PATHOGENS,
        k=1,
    )

    only_full_correlation = float(
        stats.pearsonr(
            K_only[
                upper
            ],
            K_full[
                upper
            ],
        ).statistic
    )

    minus_full_correlation = float(
        stats.pearsonr(
            K_minus[
                upper
            ],
            K_full[
                upper
            ],
        ).statistic
    )

    prepared_kernels[
        (
            group_index,
            "group_only",
        )
    ] = prepared_only

    prepared_kernels[
        (
            group_index,
            "leave_one_group_out",
        )
    ] = prepared_minus

    removal_fraction = float(
        fit_minus[
            "variance_fraction"
        ]
    )

    only_fraction = float(
        fit_only[
            "variance_fraction"
        ]
    )

    kernel_records.append(
        {
            "group_code": group_index,
            "group_name": group_name,
            "n_unitigs": int(
                group_unitig_counts[
                    group_index
                ]
            ),
            "kernel_denominator_fraction": float(
                denominator_group
                / total_denominator
            ),
            "baseline_variance_fraction": baseline_variance_fraction,
            "leave_one_group_out_variance_fraction": removal_fraction,
            "absolute_drop_after_removal": (
                baseline_variance_fraction
                - removal_fraction
            ),
            "relative_drop_after_removal": (
                (
                    baseline_variance_fraction
                    - removal_fraction
                )
                / baseline_variance_fraction
            ),
            "group_only_variance_fraction": only_fraction,
            "leave_one_group_out_vs_full_K_Pearson_r": minus_full_correlation,
            "group_only_vs_full_K_Pearson_r": only_full_correlation,
        }
    )

observed_ablation = pd.DataFrame(
    kernel_records
)

observed_ablation.to_csv(
    OBSERVED_ABLATION,
    index=False,
)

print("Observed broad ablation fits complete.")

print("\nRanked by fall in variance fraction after removal:")
display(
    observed_ablation.sort_values(
        "absolute_drop_after_removal",
        ascending=False,
    )[
        [
            "group_name",
            "n_unitigs",
            "kernel_denominator_fraction",
            "leave_one_group_out_variance_fraction",
            "absolute_drop_after_removal",
            "relative_drop_after_removal",
            "group_only_variance_fraction",
        ]
    ]
)

print("\nCell 19.6 complete.")
print("Transition: Cell 19.7 will run the same 1,000 MIC permutations for every leave-one-group-out and group-only kernel.")


In [ ]:
#@title Cell 19.7 - Permutation testing for all broad ablation kernels
# Purpose:
# Use the same 1,000 permuted phenotype orders for every broad kernel.
#
# For each group:
# - test the leave-one-group-out kernel;
# - test the group-only kernel.
#
# Benjamini-Hochberg q-values are reported separately across the 12
# leave-one-group-out tests and the 12 group-only tests.

rng = np.random.default_rng(
    PERMUTATION_SEED
)

permutation_indices = np.vstack(
    [
        rng.permutation(
            EXPECTED_PATHOGENS
        )
        for _ in range(
            N_PERMUTATIONS
        )
    ]
)

def benjamini_hochberg(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1,
            dtype=float,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]

    adjusted = np.minimum(
        adjusted,
        1.0,
    )

    result = np.empty(
        n,
        dtype=float,
    )

    result[
        order
    ] = adjusted

    return result

permutation_rows = []

permutation_start = time.time()

for group_index, group_name in enumerate(
    group_names
):
    for analysis_type in [
        "leave_one_group_out",
        "group_only",
    ]:
        prepared = prepared_kernels[
            (
                group_index,
                analysis_type,
            )
        ]

        if analysis_type == "leave_one_group_out":
            observed_fraction = float(
                observed_ablation.loc[
                    observed_ablation[
                        "group_code"
                    ]
                    == group_index,
                    "leave_one_group_out_variance_fraction",
                ].iloc[0]
            )

        else:
            observed_fraction = float(
                observed_ablation.loc[
                    observed_ablation[
                        "group_code"
                    ]
                    == group_index,
                    "group_only_variance_fraction",
                ].iloc[0]
            )

        permuted_fractions = np.empty(
            N_PERMUTATIONS,
            dtype=np.float64,
        )

        for permutation_index in range(
            N_PERMUTATIONS
        ):
            y_permuted = y[
                permutation_indices[
                    permutation_index
                ]
            ]

            fit = fit_null_reml_prepared(
                y_permuted,
                prepared,
            )

            permuted_fractions[
                permutation_index
            ] = fit[
                "variance_fraction"
            ]

        n_equal_or_greater = int(
            np.sum(
                permuted_fractions
                >= observed_fraction
            )
        )

        empirical_p = float(
            (
                1
                + n_equal_or_greater
            )
            / (
                N_PERMUTATIONS
                + 1
            )
        )

        permutation_rows.append(
            {
                "group_code": group_index,
                "group_name": group_name,
                "analysis_type": analysis_type,
                "observed_variance_fraction": observed_fraction,
                "permutations": N_PERMUTATIONS,
                "permuted_equal_or_greater": n_equal_or_greater,
                "empirical_p_value": empirical_p,
                "permutation_median": float(
                    np.median(
                        permuted_fractions
                    )
                ),
                "permutation_95th_percentile": float(
                    np.quantile(
                        permuted_fractions,
                        0.95,
                    )
                ),
            }
        )

        print(
            group_name,
            analysis_type,
            "- observed:",
            f"{observed_fraction:.6f}",
            "- p:",
            f"{empirical_p:.6f}",
        )

permutation_results = pd.DataFrame(
    permutation_rows
)

permutation_results[
    "Benjamini_Hochberg_q_value"
] = np.nan

for analysis_type in [
    "leave_one_group_out",
    "group_only",
]:
    mask = (
        permutation_results[
            "analysis_type"
        ]
        == analysis_type
    )

    permutation_results.loc[
        mask,
        "Benjamini_Hochberg_q_value",
    ] = benjamini_hochberg(
        permutation_results.loc[
            mask,
            "empirical_p_value",
        ].to_numpy()
    )

permutation_results.to_csv(
    PERMUTATION_RESULTS,
    index=False,
    compression="gzip",
)

print(
    "\nPermutation testing elapsed:",
    f"{(time.time() - permutation_start) / 60.0:.1f} minutes",
)

print("\nCell 19.7 complete.")
print("Transition: Cell 19.8 will combine the results, rank broad groups for refinement, and stop before any finer ablation.")


In [ ]:
#@title Cell 19.8 - Final QC, broad-group ranking, and stopping point
# Purpose:
# Combine observed and permutation results, rank the broad groups by the
# fall in variance fraction after removal, and identify which broad groups
# should be considered for finer ablation next.
#
# No automatic causal interpretation is made.

leave_out = (
    permutation_results.loc[
        permutation_results[
            "analysis_type"
        ]
        == "leave_one_group_out"
    ][
        [
            "group_code",
            "empirical_p_value",
            "Benjamini_Hochberg_q_value",
        ]
    ]
    .rename(
        columns={
            "empirical_p_value": "leave_one_group_out_empirical_p",
            "Benjamini_Hochberg_q_value": "leave_one_group_out_BH_q",
        }
    )
)

group_only = (
    permutation_results.loc[
        permutation_results[
            "analysis_type"
        ]
        == "group_only"
    ][
        [
            "group_code",
            "empirical_p_value",
            "Benjamini_Hochberg_q_value",
        ]
    ]
    .rename(
        columns={
            "empirical_p_value": "group_only_empirical_p",
            "Benjamini_Hochberg_q_value": "group_only_BH_q",
        }
    )
)

final_results = (
    observed_ablation
    .merge(
        group_manifest[
            [
                "group_code",
                "group_type",
                "reference_start_0_based",
                "reference_end_0_based_exclusive",
            ]
        ],
        on="group_code",
        how="left",
        validate="one_to_one",
    )
    .merge(
        leave_out,
        on="group_code",
        how="left",
        validate="one_to_one",
    )
    .merge(
        group_only,
        on="group_code",
        how="left",
        validate="one_to_one",
    )
)

assert len(
    final_results
) == N_GROUPS

assert final_results[
    [
        "leave_one_group_out_empirical_p",
        "group_only_empirical_p",
    ]
].notna().all().all()

final_results[
    "removal_rank"
] = (
    final_results[
        "absolute_drop_after_removal"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

final_results = (
    final_results
    .sort_values(
        [
            "removal_rank",
            "group_code",
        ]
    )
    .reset_index(drop=True)
)

final_results.to_csv(
    FINAL_RESULTS,
    index=False,
)

# Conservative refinement flag:
# retain the three groups producing the largest falls after removal.
# This flag is only a prioritisation device for the next notebook.
top_three_group_codes = set(
    final_results.head(
        3
    )[
        "group_code"
    ].astype(int)
)

final_results[
    "prioritise_for_finer_ablation"
] = (
    final_results[
        "group_code"
    ]
    .astype(int)
    .isin(
        top_three_group_codes
    )
)

# Rewrite with the final prioritisation column included.
final_results.to_csv(
    FINAL_RESULTS,
    index=False,
)

qc = pd.DataFrame(
    [
        {
            "pathogens": EXPECTED_PATHOGENS,
            "variable_unitigs": EXPECTED_UNITIGS,
            "broad_groups": N_GROUPS,
            "all_unitigs_assigned_once": int(
                group_manifest[
                    "n_unitigs"
                ].sum()
            )
            == EXPECTED_UNITIGS,
            "full_K_reconstruction_max_absolute_error": max_reconstruction_error,
            "full_K_reconstruction_pass": bool(
                max_reconstruction_error
                < 1e-10
            ),
            "leave_one_group_out_tests": int(
                (
                    permutation_results[
                        "analysis_type"
                    ]
                    == "leave_one_group_out"
                ).sum()
            ),
            "group_only_tests": int(
                (
                    permutation_results[
                        "analysis_type"
                    ]
                    == "group_only"
                ).sum()
            ),
            "permutations_per_test": N_PERMUTATIONS,
            "final_QC_pass": True,
        }
    ]
)

qc.to_csv(
    FINAL_QC,
    index=False,
)

completion_payload = {
    "status": "complete",
    "pathogens": EXPECTED_PATHOGENS,
    "variable_unitigs": EXPECTED_UNITIGS,
    "broad_groups": N_GROUPS,
    "baseline_variance_fraction": baseline_variance_fraction,
    "baseline_empirical_p_value": baseline_empirical_p,
    "top_three_groups_for_finer_ablation": (
        final_results.loc[
            final_results[
                "prioritise_for_finer_ablation"
            ],
            "group_name",
        ].tolist()
    ),
    "final_QC_pass": True,
}

COMPLETION_FILE.write_text(
    json.dumps(
        completion_payload,
        indent=2,
    ),
    encoding="utf-8",
)

print("Final QC: PASS")

print("\nBroad ablation results, ranked by loss of variance fraction after removal:")
display(
    final_results[
        [
            "removal_rank",
            "group_name",
            "group_type",
            "n_unitigs",
            "kernel_denominator_fraction",
            "leave_one_group_out_variance_fraction",
            "absolute_drop_after_removal",
            "relative_drop_after_removal",
            "leave_one_group_out_empirical_p",
            "group_only_variance_fraction",
            "group_only_empirical_p",
            "group_only_BH_q",
            "prioritise_for_finer_ablation",
        ]
    ]
)

print(
    "\nTop three broad groups provisionally prioritised for finer ablation:",
    ", ".join(
        final_results.loc[
            final_results[
                "prioritise_for_finer_ablation"
            ],
            "group_name",
        ]
    ),
)

print(
    "\nInterpretation rule: a larger fall after removal means that the broad "
    "group contributes more strongly to retaining the reconstructed collective "
    "association. A group-only association indicates that the group can also "
    "carry related sequence structure by itself."
)

print(
    "\nNotebook 19 stops here. Review the broad ablation pattern before "
    "splitting any group into smaller sequence sets."
)
